In [7]:
from pathlib import Path

import pandas as pd

from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap

from bst_bev.maps.bst_map_index import BSTMapIndex


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATAROOT = PROJECT_ROOT / "data" / "nuscenes"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


nusc = NuScenes(
    version="v1.0-mini",
    dataroot=str(DATAROOT),
    verbose=False,
)

scene = nusc.scene[0]

log = nusc.get(
    "log",
    scene["log_token"],
)

location = log["location"]

nusc_map = NuScenesMap(
    dataroot=str(DATAROOT),
    map_name=location,
)


sample = nusc.get(
    "sample",
    scene["first_sample_token"],
)

lidar = nusc.get(
    "sample_data",
    sample["data"]["LIDAR_TOP"],
)

ego_pose = nusc.get(
    "ego_pose",
    lidar["ego_pose_token"],
)

ego_x, ego_y, _ = ego_pose["translation"]

print("Scene:", scene["name"])
print("Location:", location)
print("Ego:", ego_x, ego_y)

Scene: scene-0061
Location: singapore-onenorth
Ego: 411.3039349319818 1180.8903791765097


In [8]:
MAP_LAYERS = [
    "lane_divider",
    "road_divider",
    "ped_crossing",
]

RADIUS = 30.0


patch = (
    ego_x - RADIUS,
    ego_y - RADIUS,
    ego_x + RADIUS,
    ego_y + RADIUS,
)

records = nusc_map.get_records_in_patch(
    patch,
    MAP_LAYERS,
    mode="intersect",
)

gt_tokens = {
    (layer, token)
    for layer, tokens in records.items()
    for token in tokens
}

print("Ground-truth objects:", len(gt_tokens))

Ground-truth objects: 4


In [9]:
def run_experiment(
    zoom,
    *,
    adaptive_index=False,
    adaptive_query=False,
):
    index = BSTMapIndex(
        nusc_map=nusc_map,
        location=location,
        zoom=zoom,
        index_spacing_m=1.0,
        adaptive_index_spacing=adaptive_index,
    )

    index.build()

    if adaptive_query:
        query = index.query_patch(
            center_x=ego_x,
            center_y=ego_y,
            radius=RADIUS,
        )
    else:
        query = index.query_patch(
            center_x=ego_x,
            center_y=ego_y,
            radius=RADIUS,
            step_m=2.0,
        )

    bst_tokens = query.tokens

    correct = bst_tokens & gt_tokens

    tp = len(correct)
    fp = len(bst_tokens - gt_tokens)
    fn = len(gt_tokens - bst_tokens)

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    return {
        "zoom": zoom,

        "recall": recall,
        "precision": precision,

        "tp": tp,
        "fp": fp,
        "fn": fn,

        "bst_candidates": len(bst_tokens),
        "ground_truth": len(gt_tokens),

        "occupied_index_cells": len(index),

        "cell_width_m": index.cell_width_at(
            ego_x,
            ego_y,
        ),

        "query_step_m": query.actual_step_m,
        "query_samples": query.sample_count,
        "cells_queried": query.cell_count,

        "index_samples": index.index_sample_count,
    }

In [10]:
def save_results(
    output_dir,
    *,
    experiment_a=None,
    experiment_b=None,
    experiment_c=None,
    comparison=None,
):
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    datasets = {
        "experiment_a.csv": experiment_a,
        "experiment_b.csv": experiment_b,
        "experiment_c.csv": experiment_c,
        "comparison.csv": comparison,
    }

    for filename, dataframe in datasets.items():
        if dataframe is not None:
            dataframe.to_csv(
                output_dir / filename,
                index=False,
            )

### fixed 1 m map indexing + fixed 2 m query sampling

In [11]:
results_a = [
    run_experiment(
        zoom,
        adaptive_index=False,
        adaptive_query=False,
    )
    for zoom in range(18, 30)
]

df_a = pd.DataFrame(results_a)
df_a

,zoom,recall,precision,tp,fp,fn,bst_candidates,ground_truth,occupied_index_cells,cell_width_m,query_step_m,query_samples,cells_queried,index_samples
0,18,1.00,0.105263,4,34,0,38,4,66,152.834780,2.0,961,2,23099
1,19,1.00,0.266667,4,11,0,15,4,187,76.417390,2.0,961,4,23099
2,20,1.00,0.307692,4,9,0,13,4,436,38.208695,2.0,961,9,23099
3,21,1.00,1.000000,4,0,0,4,4,1063,19.104347,2.0,961,20,23099
4,22,1.00,1.000000,4,0,0,4,4,2410,9.552174,2.0,961,56,23099
5,23,1.00,1.000000,4,0,0,4,4,5140,4.776087,2.0,961,196,23099
6,24,1.00,1.000000,4,0,0,4,4,10655,2.388043,2.0,961,702,23099
7,25,1.00,1.000000,4,0,0,4,4,18602,1.194022,2.0,961,961,23099
8,26,1.00,1.000000,4,0,0,4,4,22849,0.597011,2.0,961,961,23099
9,27,0.75,1.000000,3,0,1,3,4,22932,0.298505,2.0,961,961,23099


### fixed 1 m map indexing + adaptive query sampling

In [12]:
results_b = [
    run_experiment(
        zoom,
        adaptive_index=False,
        adaptive_query=True,
    )
    for zoom in range(18, 30)
]

df_b = pd.DataFrame(results_b)
df_b

,zoom,recall,precision,tp,fp,fn,bst_candidates,ground_truth,occupied_index_cells,cell_width_m,query_step_m,query_samples,cells_queried,index_samples
0,18,1.0,0.105263,4,34,0,38,4,66,152.834780,1.000000,3721,2,23099
1,19,1.0,0.266667,4,11,0,15,4,187,76.417390,1.000000,3721,4,23099
2,20,1.0,0.307692,4,9,0,13,4,436,38.208695,1.000000,3721,9,23099
3,21,1.0,1.000000,4,0,0,4,4,1063,19.104347,1.000000,3721,20,23099
4,22,1.0,1.000000,4,0,0,4,4,2410,9.552174,1.000000,3721,56,23099
5,23,1.0,1.000000,4,0,0,4,4,5140,4.776087,1.000000,3721,196,23099
6,24,1.0,1.000000,4,0,0,4,4,10655,2.388043,1.000000,3721,702,23099
7,25,1.0,1.000000,4,0,0,4,4,18602,1.194022,0.594059,10404,2652,23099
8,26,1.0,1.000000,4,0,0,4,4,22849,0.597011,0.297030,41209,10404,23099
9,27,1.0,1.000000,4,0,0,4,4,22932,0.298505,0.148883,163216,40804,23099


### adaptive map indexing + adaptive query sampling

In [13]:
results_c = [
    run_experiment(
        zoom,
        adaptive_index=True,
        adaptive_query=True,
    )
    for zoom in range(18, 30)
]

df_c = pd.DataFrame(results_c)
df_c

,zoom,recall,precision,tp,fp,fn,bst_candidates,ground_truth,occupied_index_cells,cell_width_m,query_step_m,query_samples,cells_queried,index_samples
0,18,1.0,0.105263,4,34,0,38,4,66,152.834780,1.000000,3721,2,23099
1,19,1.0,0.266667,4,11,0,15,4,187,76.417390,1.000000,3721,4,23099
2,20,1.0,0.307692,4,9,0,13,4,436,38.208695,1.000000,3721,9,23099
3,21,1.0,1.000000,4,0,0,4,4,1063,19.104347,1.000000,3721,20,23099
4,22,1.0,1.000000,4,0,0,4,4,2410,9.552174,1.000000,3721,56,23099
5,23,1.0,1.000000,4,0,0,4,4,5140,4.776087,1.000000,3721,196,23099
6,24,1.0,1.000000,4,0,0,4,4,10655,2.388043,1.000000,3721,702,23099
7,25,1.0,1.000000,4,0,0,4,4,20455,1.194022,0.594059,10404,2652,37599
8,26,1.0,1.000000,4,0,0,4,4,40565,0.597011,0.297030,41209,10404,73678
9,27,1.0,1.000000,4,0,0,4,4,81006,0.298505,0.148883,163216,40804,145796


In [15]:
save_results(
    OUTPUT_DIR,
    experiment_a=df_a,
    experiment_b=df_b,
    experiment_c=df_c,
    # comparison=comparison,
)

print(f"Results saved to: {OUTPUT_DIR}")

Results saved to: /Users/abinavbala/College/Research/BST-BEV/outputs
